In [ ]:
import sagemaker
from sklearn.model_selection import train_test_split
import boto3
import pandas as pd

region = 'eu-north-1'
boto_session = boto3.Session(region_name=region)
sm_boto3 = boto3.client('sagemaker', region_name=region)
sess = sagemaker.Session(boto_session=boto_session)
bucket = 'my-first-aibucket'
print("using bucket: " + bucket)

In [ ]:
import boto3
s3 = boto3.client('s3', region_name='eu-north-1')
s3.create_bucket(
    Bucket='my-first-aibucket',
    CreateBucketConfiguration={'LocationConstraint': 'eu-north-1'}
)

In [ ]:
df = pd.read_csv('train (1).csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df['price_range'].value_counts(normalize=False)

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.isnull().mean()* 100

In [ ]:
features = list(df.columns)
features

In [ ]:
label = features.pop(-1)
label

In [ ]:
x = df[features]
y = df[label]

In [ ]:
x.head()

In [ ]:
y.head()

In [ ]:
x.shape

In [ ]:
y.value_counts(normalize=False)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
trainX = pd.DataFrame(X_train)
trainX[label] = y_train

testX = pd.DataFrame(X_test)
testX[label] = y_test

In [ ]:
print(trainX.shape, testX.shape)

In [ ]:
trainX.head()

In [ ]:
trainX.isnull().sum()

In [ ]:
testX.isnull().sum()

In [ ]:
trainX.to_csv('train-V-1.csv', index=False)
testX.to_csv('test-V-1.csv', index=False)

### We are sending the Data to S3 Bucket ###

In [ ]:
import boto3
session = boto3.Session()
creds = session.get_credentials()
print(creds.access_key)

In [ ]:
sk_prefix = 'sagemaker/sklearn-container'
trainpath = sess.upload_data(path='train-V-1.csv', bucket=bucket, key_prefix=sk_prefix)
testpath = sess.upload_data(path='test-V-1.csv', bucket=bucket, key_prefix=sk_prefix)
print(trainpath)
print(testpath)

### We are writing the Script.py ###

In [42]:
%%writefile train.py

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
import sklearn
import joblib
import os
import numpy as np
import pandas as pd
import boto3
import pathlib
from io import StringIO
import argparse


def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir, "model.joblib"))
    return clf

if __name__ == "__main__":

    print("[INFO] Reading training data")
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_estimators", type=int, default=100)
    parser.add_argument("--random_state", type=int, default=0)

    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    parser.add_argument("--train-file", type=str, default="train-V-1.csv")
    parser.add_argument("--test-file", type=str, default="test-V-1.csv")

    args = parser.parse_known_args()

    print("SKLearn Version: ", sklearn.__version__)
    print("Joblib Version: ", joblib.__version__)
    print("[INFO] Reading training data")
    print()
    train_df = pd.read_csv(os.path.join(args[0].train, args[0].train_file))
    test_df = pd.read_csv(os.path.join(args[0].test, args[0].test_file))

    features = list(train_df.columns)
    label = features.pop(-1)

    print("Building training and testing datasets")
    print()
    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df[label]
    y_test = test_df[label]

    print('Colum order:')
    print(features)
    print()


    print("Label column: ", label)
    print()

    print("Data Shape:")
    print()
    print("---- SHAPE OF TRAINING DATA (85%) ----")
    print("X_train: ", X_train.shape)
    print("y_train: ", y_train.shape)
    print()
    print("---- SHAPE OF TESTING DATA (15%) ----")
    print("X_test: ", X_test.shape)
    print("y_test: ", y_test.shape)
    print()

    print("Traning Random Forest model....")
    print()
    model = RandomForestClassifier(n_estimators=args[0].n_estimators, random_state=args[0].random_state)
    model.fit(X_train, y_train)
    print()

    model_path = os.path.join(args[0].model_dir, "model.joblib")
    joblib.dump(model, model_path)
    print("Model persisted at location: ", model_path)
    print()


    y_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    test_rep = classification_report(y_test, y_pred)

    print()
    print("---- Metrics Results For Testing Data ----")
    print()
    print("total Rows are: ", X_test.shape[0])
    print('[TESTING] Model Accuracy: ', test_acc)
    print('[TESTING] Classification Report: ')
    print(test_rep)
    





Writing train.py


In [65]:
from sagemaker.sklearn.estimator import SKLearn
from sagemaker import get_execution_role

role = "arn:aws:iam::024882962710:role/SageMakerExecutionRole-MobileClassification"

sklearn_estimator = SKLearn(
    entry_point='train.py',
    instance_count=1,
    role=role,
    instance_type='ml.m5.large',
    framework_version='1.2-1',
    base_job_name='sklearn-rf-classifier',
    py_version='py3',
    hyperparameters={
        'n_estimators': 100,
        'random_state': 0
    },

)



In [66]:
sklearn_estimator.fit({'train': trainpath, 'test': testpath}, wait=True)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sklearn-rf-classifier-2026-09-11-09-10-14-767
ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-training-job


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 sklearn_estimator.fit({'train': trainpath, 'test': testpath}, wait=True)                     │
│   2                                                                                              │
│                                                                                                  │
│ c:\envs\myenv\Lib\site-packages\sagemaker\telemetry\telemetry_logging.py:171 in wrapper          │
│                                                                                                  │
│   168 │   │   │   │   │   caught_ex = e                                                          │
│   169 │   │   │   │   finally:                                                                   │
│   170 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 171 │   │   │   │   │   │   raise caught_ex                                                    │
│   172 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   173 │   │   │   else:                                                                          │
│   174 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ c:\envs\myenv\Lib\site-packages\sagemaker\telemetry\telemetry_logging.py:142 in wrapper          │
│                                                                                                  │
│   139 │   │   │   │   start_timer = perf_counter()                                               │
│   140 │   │   │   │   try:                                                                       │
│   141 │   │   │   │   │   # Call the original function                                           │
│ ❱ 142 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   143 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   144 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   145 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ c:\envs\myenv\Lib\site-packages\sagemaker\workflow\pipeline_context.py:346 in wrapper            │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ c:\envs\myenv\Lib\site-packages\sagemaker\estimator.py:1438 in fit                               │
│                                                                                                  │
│   1435 │   │   self._prepare_for_training(job_name=job_name)                                     │
│   1436 │   │                                               

### Deployment Process ###